# Day 2 — Silver Layer Practice Questions
## Topic: PySpark string cleaning · Type casting · Derived columns · Window functions

---

> **Rules:**
> - Run Day 1 first so bronze tables exist
> - All solutions must use **PySpark** — no pandas transforms
> - Solve each question before peeking at the hint

| # | Difficulty | Topic |
|---|-----------|-------|
| Q1 | 🟢 Easy | PySpark: clean products — trim, upper, cast, dedup |
| Q2 | 🟡 Medium | PySpark: build order value tier with F.when().otherwise() |
| Q3 | 🔴 Hard | PySpark: dense_rank + lag window functions on order_items |

---
## Setup — run once

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from config.db_config import (
    get_engine, set_spark_env, get_spark, new_batch,
    spark_read_pg, spark_write_pg
)

engine = get_engine()
_, SILVER_AT = new_batch()

set_spark_env()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, BooleanType

spark = get_spark('Day2_PQ')

def read_bronze(table):
    return spark_read_pg(spark, 'bronze', table)

print('Setup ready.')

---
---
## Q1 — 🟢 Easy
### PySpark: clean bronze.products and write to silver

---

### Problem Statement

1. Read `bronze.products` via `read_bronze('products')` → PySpark DataFrame
2. Apply the following with PySpark column expressions:
   - `product_name`, `brand`: `F.trim()` whitespace
   - `category`: `F.upper(F.trim())`
   - `unit_price`: `.cast(DoubleType())`
   - `is_available`: `F.when(F.lower().isin('true','1'), True).otherwise(False).cast(BooleanType())`
3. Deduplicate on `product_id` using `.dropDuplicates(['product_id'])`
4. Drop all `_` metadata cols: `.drop(*[c for c in df.columns if c.startswith('_')])`
5. Add `_silver_loaded_at` via `F.lit(SILVER_AT)`
6. Write to `silver.pq_products` using `spark_write_pg(df, 'silver', 'pq_products')`
7. Show unique categories

### Expected: 5 unique categories (all caps)
```
['ELECTRONICS', 'FOOTWEAR', 'KITCHEN', 'NUTRITION', 'SPORTS']
```

**Hints:** `F.trim(F.col('col'))`, `F.upper(...)`, `.cast(DoubleType())`, `F.lower().isin(...)`

In [ ]:
# Q1 Setup
df_prod = read_bronze('products')
print(f'bronze.products: {df_prod.count()} rows')
df_prod.select('product_id','product_name','category','unit_price','is_available').show(3)

In [ ]:
# Q1 — Write your solution here


<details><summary>💡 Solution</summary>

```python
df = (df_prod
    .withColumn('product_name', F.trim(F.col('product_name')))
    .withColumn('brand',        F.trim(F.col('brand')))
    .withColumn('category',     F.upper(F.trim(F.col('category'))))
    .withColumn('unit_price',   F.col('unit_price').cast(DoubleType()))
    .withColumn('is_available',
        F.when(F.lower(F.col('is_available')).isin('true','1'), F.lit(True))
         .otherwise(F.lit(False))
         .cast(BooleanType()))
    .dropDuplicates(['product_id'])
)

meta_cols = [c for c in df.columns if c.startswith('_')]
df = df.drop(*meta_cols)
df = df.withColumn('_silver_loaded_at', F.lit(SILVER_AT))

spark_write_pg(df, 'silver', 'pq_products')

cats = sorted([r[0] for r in df.select('category').distinct().collect()])
print('Unique categories:', cats)
```
</details>

---
---
## Q2 — 🟡 Medium
### PySpark: build order value tier with F.when().otherwise()

---

### Problem Statement

1. Read `bronze.orders` via `read_bronze('orders')` → PySpark DataFrame
2. Cast: `total_amount` → `DoubleType()`, `order_date` → `DateType()` using `F.to_date()`
3. Add `is_cancelled`: `F.when(F.lower(F.col('status')) == 'cancelled', True).otherwise(False).cast(BooleanType())`
4. Add `order_value_tier` using nested `F.when().when().otherwise()`:
   - `'high'` if `total_amount >= 200`
   - `'medium'` if `total_amount >= 100`
   - `'low'` otherwise
5. Show tier distribution using `.groupBy('order_value_tier').count().show()`
6. Drop `_` metadata cols, add `_silver_loaded_at`
7. Write to `silver.pq_orders` using `spark_write_pg(df, 'silver', 'pq_orders')`

### Expected tier distribution (approximate)
```
+----------------+-----+
|order_value_tier|count|
+----------------+-----+
|          medium|   23|
|             low|   15|
|            high|   12|
+----------------+-----+
```

**Hints:** `F.when(cond, val).when(cond2, val2).otherwise(val3)`, `F.to_date(F.col('col'), 'yyyy-MM-dd')`

In [ ]:
# Q2 Setup
df_ord = read_bronze('orders')
print(f'bronze.orders: {df_ord.count()} rows')
df_ord.select('order_id','status','total_amount').show(3)

In [ ]:
# Q2 — Write your solution here


<details><summary>💡 Solution</summary>

```python
from pyspark.sql.types import DateType

df = (df_ord
    .withColumn('total_amount', F.col('total_amount').cast(DoubleType()))
    .withColumn('order_date',   F.to_date(F.col('order_date'), 'yyyy-MM-dd'))
    .withColumn('is_cancelled',
        F.when(F.lower(F.col('status')) == 'cancelled', F.lit(True))
         .otherwise(F.lit(False))
         .cast(BooleanType()))
    .withColumn('order_value_tier',
        F.when(F.col('total_amount') >= 200, F.lit('high'))
         .when(F.col('total_amount') >= 100, F.lit('medium'))
         .otherwise(F.lit('low')))
)

print('Tier distribution:')
df.groupBy('order_value_tier').count().show()

meta = [c for c in df.columns if c.startswith('_')]
df = df.drop(*meta).withColumn('_silver_loaded_at', F.lit(SILVER_AT))

spark_write_pg(df, 'silver', 'pq_orders')
print(f'silver.pq_orders: {df.count()} rows')
```
</details>

---
---
## Q3 — 🔴 Hard
### PySpark: DENSE_RANK and LAG window functions on order_items

---

### Problem Statement

Using PySpark on `data/raw/order_items.csv`:

1. Read the CSV with `.option('header','true')` and cast types:
   - `quantity` → `IntegerType()`
   - `unit_price`, `discount_pct` → `DoubleType()`
   - Fill null `discount_pct` with `0.0` using `.fillna({'discount_pct': 0.0})`
2. Compute `line_total = quantity * unit_price * (1 - discount_pct/100)` rounded to 2dp
3. Add `dense_rank_in_order` — `F.dense_rank()` within each `order_id` ordered by `line_total` desc
   - Note: `dense_rank` has no gaps (1,1,2 not 1,1,3 like `rank`)
4. Add `prev_item_total` — `F.lag('line_total', 1)` within each `order_id` ordered by `item_id`
5. Add `diff_from_prev` — `line_total - prev_item_total` (null for first item)
6. Filter to order `O001` and show: `item_id`, `line_total`, `dense_rank_in_order`, `prev_item_total`, `diff_from_prev`

**Hints:** `F.dense_rank()`, `F.lag('col', 1).over(window)`, `Window.partitionBy().orderBy()`, `F.coalesce()` not needed — lag gives null automatically

In [ ]:
# Q3 Setup
df_items_raw = spark.read.option('header','true').csv(raw_path('order_items.csv'))
print(f'order_items rows: {df_items_raw.count()}')
df_items_raw.printSchema()

In [ ]:
# Q3 — Write your solution here


<details><summary>💡 Solution</summary>

```python
df = (df_items_raw
    .withColumn('quantity',     F.col('quantity').cast(IntegerType()))
    .withColumn('unit_price',   F.col('unit_price').cast(DoubleType()))
    .withColumn('discount_pct', F.col('discount_pct').cast(DoubleType()))
    .fillna({'discount_pct': 0.0})
)

df = df.withColumn(
    'line_total',
    F.round(F.col('quantity') * F.col('unit_price') * (1 - F.col('discount_pct') / 100), 2)
)

# DENSE_RANK: no gaps when values tie (compare to rank which skips)
w_rank = Window.partitionBy('order_id').orderBy(F.col('line_total').desc())
df = df.withColumn('dense_rank_in_order', F.dense_rank().over(w_rank))

# LAG: get the previous row's value within same order
w_lag = Window.partitionBy('order_id').orderBy('item_id')
df = df.withColumn('prev_item_total', F.lag('line_total', 1).over(w_lag))
df = df.withColumn('diff_from_prev', F.round(F.col('line_total') - F.col('prev_item_total'), 2))

df.filter(F.col('order_id') == 'O001') \
  .select('item_id','order_id','line_total','dense_rank_in_order','prev_item_total','diff_from_prev') \
  .orderBy('item_id') \
  .show(truncate=False)
```
</details>

In [ ]:
spark.stop()
print('Spark stopped.')